# all-reduce-grad-sync — faded example 2: Complete the all_reduce SUM step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-grad-sync`. Running the beacon reports progress on the `Distributed: all_reduce grad sync` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce grad sync` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-grad-sync`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-grad-sync"
DD_SUBTOPIC = "Distributed: all_reduce grad sync"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The first half of grad sync is the collective itself: `all_reduce(grad, op=SUM)` replaces every rank's grad buffer with the element-wise sum across all ranks, in place. Only after this SUM is shared can each rank divide by world_size to get the mean.

## Faded exercise 2

Two ranks hold grads `[2, 4]` and `[6, 8]`. Complete `mock_all_reduce_sum` so it performs the SUM all-reduce: every buffer must end up holding the element-wise sum across both ranks, in place. The mean divide afterwards is already written.

**Fill in:** computing the element-wise sum across all rank buffers and copying it back into each buffer in place.

In [ ]:
import numpy as np
import torch as t

t.manual_seed(0)
world_size = 2
grads = [t.tensor([2.0, 4.0]), t.tensor([6.0, 8.0])]

def mock_all_reduce_sum(buffers):
    total = sum(b.clone() for b in buffers)
    for b in buffers:
        b.copy_(total)

mock_all_reduce_sum(grads)
for g in grads:
    g /= world_size


def _test():
    import torch as t
    expected = t.tensor([4.0, 6.0])  # mean of [2,4] and [6,8]
    assert t.allclose(grads[0], expected), grads[0]
    assert t.allclose(grads[1], expected), grads[1]
    assert t.allclose(grads[0], grads[1]), 'ranks must agree after sync'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
import torch as t

t.manual_seed(0)
world_size = 2
grads = [t.tensor([2.0, 4.0]), t.tensor([6.0, 8.0])]

def mock_all_reduce_sum(buffers):
    total = sum(b.clone() for b in buffers)
    for b in buffers:
        b.copy_(total)

mock_all_reduce_sum(grads)
for g in grads:
    g /= world_size
```
</details>